# 00 — Orientation: the course, the shop, your first live call

**What you'll learn**

- Make your first live model call with `litellm.completion` and read the answer, the token counts, and the exact dollar cost off the response
- Rerun everything for free: how the disk cache in `.litellm_cache/` works and how to watch it hit
- Load the Larkspur Outfitters micro-world with the `shoplab.world` loaders and poke at it with pandas
- Build a tiny IDF-weighted policy retriever by hand, then meet its canonical form: `shoplab.world.search_policy`
- See the gold tickets the whole course is graded against, and the three-field answer every agent must produce

*Time: ~15 min. Cost: under $0.01. Cached reruns are free, minus one call that deliberately skips the cache.*

## The course in one idea

Most agent courses start with a framework and work down. This one starts at the bottom and stays there for ten chapters: you build the tool loop, the tracing layer, the eval harness, the budget and approval gates, and the context-management machinery yourself, in plain Python, each piece small enough to read in one sitting. The frameworks — LangGraph, smolagents, MCP, A2A, and friends — arrive in the second half as what they actually are: opinionated packagings of parts you already own. The framing follows Anthropic's [*Building Effective AI Agents*](https://www.anthropic.com/engineering/building-effective-agents), which draws the useful line between workflows — LLMs and tools orchestrated through predefined code paths — and agents, which direct their own tool use; this course builds both and stays precise about which is which.

Nothing here is a paper exercise. Every chapter makes live model calls, measures what they cost, and grades outputs against fixed gold labels in one persistent micro-world. By the end of chapter 10 there is a package — `shoplab` — whose every important function you first wrote in a notebook.

## Setup

Mirrors the README, once, so this notebook is self-contained. Requires Python 3.12 or newer.

```bash
python -m venv .venv && source .venv/bin/activate
pip install -e ".[obs]"
cp .env.example .env    # then add your OPENROUTER_API_KEY
jupyter lab
```

One secret is required: `OPENROUTER_API_KEY`. `MODEL` and `STRONG_MODEL` are LiteLLM provider strings with working defaults; leave them unset until you have a reason not to. The `.env` contract is identical to the sibling course [dspy-lab](https://github.com/sinaghadermarzi/dspy-lab), so one `.env` file serves both repos.

The next cell is the course's frozen config — byte-identical as the first code cell of every notebook. Run it and move on.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

Chapter 03 builds tracing by hand; until then, this frozen cell finds or starts a local [Phoenix](https://arize.com/docs/phoenix) server and routes every LiteLLM call to it, so there is a trace of everything you run today waiting for you when tracing becomes the topic. Optional — skip it and nothing else in this chapter changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## Your first live call

The entire course talks to models through one function: `litellm.completion`. It takes a provider-prefixed model string plus OpenAI-shaped messages and returns an OpenAI-shaped response, whatever backend actually served it. Beyond the answer itself, two fields matter: `resp.usage` carries the token counts, and `resp._hidden_params["response_cost"]` carries LiteLLM's per-call cost accounting, in dollars.

Ask it an ops question.

In [ ]:
resp = litellm.completion(
    model=MODEL,
    temperature=TEMPERATURE,
    messages=[{"role": "user", "content":
               "In one sentence: what does the ops desk of a small online shop do all day?"}],
)
print(resp.choices[0].message.content)
print(f"tokens: {resp.usage.prompt_tokens} in, {resp.usage.completion_tokens} out")
print(f"cost:   ${resp._hidden_params['response_cost']:.8f}")

> **What you should see:** a one-sentence answer about orders, stock, shipping, and unhappy customers — the wording varies between runs and providers, and `temperature=0` does not change that. Token counts in the tens each way, and a cost measured in thousandths of a cent. If the call fails instead, check `OPENROUTER_API_KEY` in your `.env`.

## Reruns are free

The config cell pointed LiteLLM at a disk cache. An identical call — same model, same messages, same parameters — is answered from `.litellm_cache/` instead of the network: no latency, no charge. That mechanic is what makes this course cheap to rerun; only prompts you have never sent before cost money.

Prove it with a stopwatch — honestly. On a rerun of this notebook both calls below would already be cached and the stopwatch would show no contrast at all, so the first call opts out of reading the cache with `cache={"no-cache": True}`: it always pays the network price (a fraction of a cent) and still writes what it gets back to disk, which is exactly the entry the second call then hits.

In [ ]:
import time

def timed_call(prompt, **kwargs):
    t0 = time.time()
    r = litellm.completion(model=MODEL, temperature=TEMPERATURE,
                           messages=[{"role": "user", "content": prompt}], **kwargs)
    return time.time() - t0, r._hidden_params.get("cache_hit")

question = "Name the three worst things that can happen to a parcel."
for attempt, kwargs in ((1, {"cache": {"no-cache": True}}), (2, {})):
    seconds, hit = timed_call(question, **kwargs)
    print(f"call {attempt}: {seconds:7.3f}s   cache_hit={hit}")

> **What you should see:** call 1 goes over the network — seconds to tens of seconds, `cache_hit=None` — and call 2 answers from disk in milliseconds with `cache_hit=True`. That gap of three to four orders of magnitude is the whole demonstration, and the second number is the one this course leans on. Delete `.litellm_cache/` when you want everything to go back to paying.

## Meet Larkspur Outfitters

Every chapter runs against the same micro-world: Larkspur Outfitters, a small online outdoor-gear shop, seen from its back office — the ops desk. The data is hand-authored JSON in `data/`, about 60 KB in total, and every product name, SKU, and customer is invented, so no model can substitute memorized priors for actually looking things up.

The desk's recurring job — and the spine of this course — is triaging return and refund tickets. Every ticket resolves to exactly three fields: a **decision** (refund, partial refund, replacement, store credit, deny, or escalate), the **policy id** that justifies it, and the **amount** in dollars. Chapters differ in how that answer gets produced; the answer itself never changes shape.

In [ ]:
import pandas as pd
from shoplab import world

products = world.load_products()
customers = world.load_customers()
orders = world.load_orders()
print(f"{len(products)} products, {len(customers)} customers, {len(orders)} orders")

catalog = pd.DataFrame(products)
print(catalog[["sku", "name", "category", "price_usd", "stock"]].head(3).to_string(index=False))
print(f"prices run ${catalog.price_usd.min():.2f} to ${catalog.price_usd.max():.2f}")

> **What you should see:** 30 products, 15 customers, 40 orders. SKUs run `LK-1001` through `LK-1030` and prices span single digits to a few hundred dollars — a world small enough to hold in your head, but big enough that an agent has to look things up rather than guess.

## The policy shelf

Twelve short policy documents govern the desk: return windows, restocking fees, damage claims, battery shipping, and the rest. Each is under 180 words and states the exact parameters the shop operates by — numbers you will meet again in chapter 04, when the rules engine that uses them appears. Agents are expected to cite policies by id, so start by learning the shelf.

In [ ]:
from shoplab.world import load_policies

policies = load_policies()
for p in policies:
    print(f"{p['id']:<18} {p['title']}")

An agent that cites policy first has to find policy. Twelve documents do not justify an embedding model; keyword search with IDF weighting is deterministic, inspectable end to end, and — chapter 04 will put a number on this — good enough. Build it in two steps.

Step one: lowercase, split into word tokens, and drop stopwords — the little words that carry no signal about which policy is meant.

In [ ]:
import re

query = "Refund for an opened box -- the tent is fine, I just do not need it."
raw = re.findall(r"[a-z0-9']+", query.lower())
stop = {"a", "an", "and", "the", "is", "for", "to", "it", "i", "of"}
print("raw: ", raw)
print("kept:", [t for t in raw if t not in stop])

Step two: weigh the surviving tokens. A word that appears in almost every policy (`refund`) says little about which one you want; a word that appears in one (`battery`) says nearly everything. Inverse document frequency — `log(N / df)` where `df` counts the documents containing the word — turns that intuition into a number.

In [ ]:
import math

doc_tokens = [set(re.findall(r"[a-z0-9']+", (p["title"] + " " + p["text"]).lower()))
              for p in policies]
df = {}
for tokens in doc_tokens:
    for tok in tokens:
        df[tok] = df.get(tok, 0) + 1
for word in ["the", "refund", "restocking", "battery"]:
    print(f"{word:<12} in {df[word]:>2}/12 docs   idf = {math.log(12 / df[word]):.2f}")

> **What you should see:** `the` appears in all 12 documents and scores exactly 0.0; `refund` is nearly everywhere, so it is worth little; `restocking` and `battery` are rare and carry real weight. A query's score against a document is simply the summed IDF of the tokens they share.

That is the whole algorithm. The next cell is the finished form — and it is not a transcript. This exact text, sentinel comments included, lives in `src/shoplab/world.py`; the course validator byte-compares the two, so the notebook can never drift from the package. From chapter 01 on, notebooks just write `from shoplab.world import search_policy`.

In [ ]:
# >>> shoplab.world.search_policy
_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from",
    "had", "has", "have", "i", "in", "is", "it", "my", "of", "on", "or",
    "that", "the", "this", "to", "was", "we", "with", "you",
}


def _tokenize(text):
    return [t for t in re.findall(r"[a-z0-9']+", text.lower()) if t not in _STOPWORDS]


def search_policy(query, k=2):
    """Return the k policy docs best matching `query` (IDF-weighted token overlap)."""
    policies = load_policies()
    doc_tokens = [set(_tokenize(p["title"] + " " + p["text"])) for p in policies]
    df = {}
    for tokens in doc_tokens:
        for tok in tokens:
            df[tok] = df.get(tok, 0) + 1
    idf = {tok: math.log(len(policies) / n) for tok, n in df.items()}
    q = set(_tokenize(query))
    scored = [(sum(idf[t] for t in q & tokens), p)
              for p, tokens in zip(policies, doc_tokens)]
    scored.sort(key=lambda pair: -pair[0])      # stable: ties keep policy order
    return [{"id": p["id"], "title": p["title"], "text": p["text"],
             "score": round(score, 4)}
            for score, p in scored[:k]]
# <<< shoplab.world.search_policy

In [ ]:
for query in ["refund for an opened box", "arrived cracked"]:
    hits = search_policy(query)
    print(query, "->", [(h["id"], h["score"]) for h in hits])

> **What you should see:** `refund for an opened box` puts `pol-restocking` in its top two; `arrived cracked` ranks `pol-damaged` first. The ids come back identical on every run — this is arithmetic over fixed files, no model in the loop — which is exactly what you want from the retrieval layer under a system whose other parts are stochastic.

## The gold tickets

`data/tickets.json` holds 40 return/refund tickets, pre-split: 20 train, 12 dev, 8 test. Each ticket references a delivered order and carries a `gold` answer — decision, policy id, amount. The labels are not hand-assigned opinions: a deterministic rules cascade (`shoplab.rules.decide`, built in chapter 04) computes every one, and the repo's data checks recompute them on every build.

Those 40 answers are the course's measuring stick. Chapter 04 builds the eval harness that grades any agent against them, and every architecture after that is judged by the same score. One guarantee worth knowing today: for every train ticket, the gold policy id appears in `search_policy(reason_text, k=2)` — on the train split, retrieval is never the excuse.

In [ ]:
import json
from shoplab.world import load_tickets

tickets = load_tickets()
print({split: len(rows) for split, rows in tickets.items()})
print(json.dumps(tickets["train"][0], indent=2))

> **What you should see:** splits of 20/12/8. One ticket ties together an order, a customer, a SKU, a free-text `reason_text`, and a structured `gold` block with the three fields from the job description: `decision`, `policy_id`, `refund_usd`. Whatever an agent says in later chapters gets reduced to those three fields and compared.

## What the whole course costs

Live calls all the way through, and still a smaller commitment than it sounds: a full cold run of every chapter lands around $2.50-$4.00 at the default models, and the disk cache makes reruns free. Two pieces of plumbing keep it that simple. OpenRouter exposes hundreds of models behind a single API endpoint, so one `OPENROUTER_API_KEY` covers the entire course ([OpenRouter quickstart](https://openrouter.ai/docs/quickstart)). LiteLLM selects the backend from the provider prefix of the model string, which is why `MODEL` reads `openrouter/deepseek/...` and why pointing the whole course at another provider is a one-line edit ([LiteLLM providers](https://docs.litellm.ai/docs/providers)).

## One last call: ask the model about Larkspur

Everything above came out of JSON files the model has never seen — that was the point of inventing the shop. So close the loop between this chapter's two halves. The catalog answers the next question in one line; ask the model instead.

In [ ]:
resp = litellm.completion(
    model=MODEL,
    temperature=TEMPERATURE,
    messages=[{"role": "user", "content":
               "In one sentence: what does the Ridgeline 2P Tent cost at Larkspur Outfitters?"}],
)
print(resp.choices[0].message.content)
print(f"catalog says: ${catalog.loc[catalog.sku == 'LK-1001', 'price_usd'].item():.2f}")

> **What you should see:** a fluent, confident sentence — our frozen run quotes $349.99 — against the catalog's actual 289.00. Some models hedge instead; none can look it up. Hold onto that failure: it is not a formatting problem, no parser fixes it, and chapter 01 is about meeting it up close.

## Recap

| Concept | One-liner |
|---|---|
| `litellm.completion` | One function for every provider; `MODEL` is just a provider-prefixed string. |
| Cost per call | `resp._hidden_params["response_cost"]`, in dollars; the default model runs thousandths of a cent. |
| Disk cache | `.litellm_cache/` answers repeated identical calls locally, so reruns are free. |
| Larkspur Outfitters | 30 products, 15 customers, 40 orders, 12 policies — hand-authored, invented, in `data/`. |
| The ops-desk job | Triage tickets to a three-field answer: decision, policy id, amount. |
| `search_policy` | IDF-weighted keyword search over the 12 policies; deterministic; canonical home `shoplab.world`. |
| Gold tickets | 40 labeled tickets split 20/12/8; chapter 04 grades everything against them. |

## Exercises

1. Uncomment one of the `MODEL` override lines in the config cell (any LiteLLM provider string works) and rerun the first-call cell. Compare token counts and `response_cost` across models — and explain why the rerun was not free. (Live calls: still fractions of a cent.)
2. Ask `search_policy` a hazmat question — say, "can I return a lithium battery power bank?" — and account for the ranking: which query token carries the most IDF weight, and why does the runner-up policy score at all? The `df` dictionary from this chapter answers both.
3. With pandas, find the most expensive **delivered** order in `orders`. Check whether the answer is unique — it is not — and decide how you would break the tie before an agent has to.

**Next up:** chapter 01 builds the LLM layer: one `complete` function that logs every token and dollar it spends, and `parse_json_loose` for models that wrap their JSON in fences and prose.